In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
reply = llm.invoke("In one sentence, what is LangChain?")
print(reply.content)

[{'type': 'text', 'text': 'LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs) by providing modular components for chaining data, prompts, and external tools together.', 'extras': {'signature': 'El4KXAERTTIP0WFglVh2W9uCMQo0nh1KoAbG6WdQC7N098b6RJgWvdHmJPx66Y2VDeDdUOTjFIR+XfZVJOd4V+j304PiIrkb43B52mUGewfwP+r5ogQZXDSodcJHthDd'}}]


In [5]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

In [6]:
@tool
def get_weather(city: str) -> str:
    """Return a fictional weather report for a city."""
    weather = {
        "Seattle": "Rainy, 12°C",
        "New York": "Sunny, 25°C",
        "London": "Cloudy, 15°C"
    }
    return weather.get(city, "Weather data not available")

In [7]:
llm_with_tools = llm.bind_tools([
    get_share_price,
    get_weather
])

In [9]:
conversation = [
    HumanMessage(
        content="What is Amazon's share price and what is the weather in Seattle?"
    )
]

ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls:

    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])

    elif call["name"] == "get_weather":
        result = get_weather.invoke(call["args"])

    conversation.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"],
            name=call["name"]
        )
    )

final = llm_with_tools.invoke(conversation)

print(final.content)

[{'type': 'text', 'text': "Amazon's current share price is $198. The weather in Seattle is rainy and 12°C.", 'extras': {'signature': 'El4KXAERTTIPKoSRi5vc7vvyhOr8s7iA1cb+FVTqfBI6NSUSTfmC7x/7VI07uy/8lD6PpgehVbCh7gHZ8VJwxOHmcoWrznWukCJ8M+9hnfJM2Sbyt1Kii0JdKnXZTyIh'}}]
